In [ ]:
# Module 1 - Lab 1: Memory-Based vs Generator-Based Data Loading

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os

# ---------------- Memory-Based Loading ----------------
print("=== Memory-Based Loading ===")
# Example: Load small dataset into RAM
def load_images_to_memory(data_dir, img_size=(128,128)):
    images, labels = [], []
    for class_name in os.listdir(data_dir):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir): continue
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = plt.imread(img_path)
            img = np.resize(img, (*img_size, 3))  # Resize
            images.append(img)
            labels.append(class_name)
    return np.array(images), np.array(labels)

# Usage example:
# X_train, y_train = load_images_to_memory('train_data')

# ---------------- Generator-Based Loading (Keras) ----------------
print("\n=== Generator-Based (Keras) ===")
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    'train_data',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

# ---------------- PyTorch Dataset + DataLoader ----------------
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.image_paths = []
        self.labels = []
        for label, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_dir):
                self.image_paths.append(os.path.join(class_dir, img_name))
                self.labels.append(label)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = plt.imread(self.image_paths[idx])
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = CustomImageDataset('train_data', transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print("All three approaches ready!")

In [ ]:
# Module 1 - Lab 2: Data Loading + Augmentation with Keras

from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Strong augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'path/to/train',
    target_size=(224, 224),   # Common size for transfer learning
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    'path/to/validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Visualize augmentations
def plot_augmented_images(generator):
    imgs, labels = next(generator)
    plt.figure(figsize=(12, 8))
    for i in range(9):
        plt.subplot(3, 3, i+1)
        plt.imshow(imgs[i])
        plt.axis('off')
    plt.show()

plot_augmented_images(train_generator)

In [ ]:
# Module 1 - Lab 3: Data Loading + Augmentation with PyTorch

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Advanced transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Datasets
train_dataset = datasets.ImageFolder('path/to/train', transform=train_transform)
val_dataset = datasets.ImageFolder('path/to/validation', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(f"Training samples: {len(train_dataset)} | Classes: {train_dataset.classes}")


In [ ]:
# Module 2 - Lab 1: Train and Evaluate a Keras-Based Classifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'train_data',           # Change to your training folder
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    'train_data',
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Model Definition
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Training
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator
)

# Evaluation & Plots
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.show()

# Save model
model.save('keras_classifier.h5')
print("Keras model training completed!")

In [ ]:
# Module 2 - Lab 2: Implement and Test a PyTorch-Based Classifier

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Transforms
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Datasets
train_dataset = datasets.ImageFolder('train_data', transform=transform)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=4)

# CNN Model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 18 * 18, 512),  # Adjust size based on input
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=len(train_dataset.classes))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
def train_model(epochs=15):
    train_acc, val_acc = [], []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_acc.append(100. * correct / total)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Acc: {train_acc[-1]:.2f}%')
    
    torch.save(model.state_dict(), 'pytorch_classifier.pth')
    print("PyTorch model saved!")

train_model()

In [ ]:
# Module 2 - Lab 3: Comparative Analysis

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Example results (replace with your actual results)
comparison = {
    'Framework': ['Keras/TensorFlow', 'PyTorch'],
    'Final_Accuracy': [92.5, 91.8],
    'Final_Val_Accuracy': [88.7, 89.2],
    'Training_Time_min': [45, 38],
    'Parameters_M': [8.9, 8.7],
    'Ease_of_Use': [9.0, 8.5]   # Subjective score
}

df = pd.DataFrame(comparison)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.barplot(x='Framework', y='Final_Val_Accuracy', data=df, ax=axes[0,0])
axes[0,0].set_title('Validation Accuracy Comparison')

sns.barplot(x='Framework', y='Training_Time_min', data=df, ax=axes[0,1])
axes[0,1].set_title('Training Time (minutes)')

sns.barplot(x='Framework', y='Parameters_M', data=df, ax=axes[1,0])
axes[1,0].set_title('Number of Parameters (Millions)')

# Add your own observations
print("=== Comparative Analysis ===")
print(df)

print("\nObservations:")
print("• PyTorch was slightly faster to train")
print("• Keras had simpler high-level API")
print("• Both achieved very similar accuracy")
print("• PyTorch offers more flexibility for custom layers")

In [ ]:
# Module 3 - Lab 1: Vision Transformers in Keras

import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflow_addons as tfa  # pip install tensorflow-addons
import numpy as np

# Simple Vision Transformer Block
class ViTBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout)
        ])
    
    def call(self, x, training=False):
        # Attention block
        x_norm = self.norm1(x)
        attn_output = self.att(x_norm, x_norm)
        x = x + attn_output
        
        # MLP block
        x_norm = self.norm2(x)
        mlp_output = self.mlp(x_norm, training=training)
        return x + mlp_output

# Full Vision Transformer Model
def create_vit_model(input_shape=(224, 224, 3), num_classes=10, 
                     patch_size=16, embed_dim=192, depth=8, num_heads=3):
    
    inputs = layers.Input(shape=input_shape)
    
    # Patch Embedding
    patches = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding='valid')(inputs)
    patches = layers.Reshape((-1, embed_dim))(patches)
    
    # Class Token + Positional Embedding
    batch_size = tf.shape(patches)[0]
    cls_token = tf.zeros((batch_size, 1, embed_dim))  # Learnable class token
    cls_token = layers.Dense(embed_dim)(cls_token)   # Make it learnable
    patches = tf.concat([cls_token, patches], axis=1)
    
    positions = tf.range(1 + patches.shape[1])
    pos_embedding = layers.Embedding(input_dim=1 + patches.shape[1], output_dim=embed_dim)(positions)
    patches = patches + pos_embedding
    
    # Transformer Blocks
    for _ in range(depth):
        patches = ViTBlock(embed_dim, num_heads, mlp_dim=embed_dim*4)(patches)
    
    # Classification Head
    x = layers.LayerNormalization(epsilon=1e-6)(patches[:, 0])  # Use class token
    x = layers.Dense(512, activation='gelu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    return model

model = create_vit_model(num_classes=5)  # Change num_classes to your dataset
model.compile(optimizer=tfa.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# Module 3 - Lab 2: Vision Transformers in PyTorch

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.proj(x)          # (B, embed_dim, H, W)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)
        return x

class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, embed_dim=192, depth=8, 
                 num_heads=3, num_classes=5):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4,
                activation='gelu', batch_first=True, dropout=0.1
            ) for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        return self.head(x[:, 0])   # Class token output

# Usage
model = ViT(num_classes=5)   # Change to your number of classes
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example training setup (same as previous PyTorch lab)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

In [ ]:
# Module 3 - Lab 3: CNN + Transformer Hybrid for Land Classification

import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_Transformer_Hybrid(nn.Module):
    def __init__(self, num_classes=6):  # e.g., Forest, Desert, Water, Urban, etc.
        super().__init__()
        
        # CNN Backbone (Feature Extractor)
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((14, 14))   # Fixed size for transformer
        )
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256, nhead=8, dim_feedforward=1024, 
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.cnn(x)                    # (B, 256, 14, 14)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)   # (B, H*W, C) for transformer
        x = self.transformer(x)
        x = x.transpose(1, 2).reshape(B, C, H, W)
        return self.classifier(x)

# Evaluation & Comparison
def evaluate_model(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

print("Hybrid CNN-Transformer model ready for Land Classification!")